# 📦 Optimisation Multi-Étages de la Valeur de Stock — Modèle MILP

---

## Contexte et Problématique

Dans un contexte industriel concurrentiel, la gestion des stocks représente un enjeu financier majeur : **trop de stock immobilise du capital** et génère des risques d'obsolescence, tandis que **trop peu de stock expose à des ruptures** et dégrade le taux de service client.

Ce projet s'inscrit dans une démarche concrète d'amélioration de la performance supply chain d'un site industriel multi-étages (Matières Premières → Semi-Finis → Produits Finis). L'analyse des indicateurs de performance (KPIs) révèle un **surstockage structurel** :

- 💰 **Capital immobilisé excessif** : la valeur totale du stock dépasse significativement la cible fiscale de fin d'année.
- 📉 **Rotation insuffisante** : certaines références accumulent des niveaux bien supérieurs à leur stock de sécurité optimal.
- ⚠️ **Risque d'obsolescence** : un stock trop élevé sur des références à faible rotation peut entraîner des provisions financières.

---

## Objectif du Modèle

L'objectif est de construire un **outil d'aide à la décision** basé sur la Programmation Linéaire Mixte en Nombres Entiers (**MILP — Mixed Integer Linear Programming**) capable de répondre à trois questions opérationnelles :

| Question | Levier |
|----------|--------|
| **Quand commander ?** (MP) | Variable binaire $x_{i,t}$ |
| **Combien commander / produire ?** | Variables continues $O_{i,t}^{MP}$, $P_{j,t}^{SF}$, $P_{l,t}^{PF}$ |
| **La cible fiscale est-elle atteignable ?** | Contrainte $C5$ + statut du solveur |

---

## Architecture du Modèle

Le modèle est structuré en **deux couches** :

1. **Couche théorique — EOQ** : calcule les paramètres économiques de référence (quantité économique de commande, stock de sécurité) qui calibrent le modèle MILP.
2. **Couche décisionnelle — MILP multi-périodes, multi-étages** : optimise la trajectoire de stock de juin à septembre, sous toutes les contraintes opérationnelles réelles (lead times, temps de cycle, BOM, MOQ, MLS, fermeture fournisseur en août).

```
MP ──(achat, lead time LT_i)──► Stock MP
Stock MP ──(production, BOM a_ij)──► Stock SF    (temps de cycle ct_j)
Stock SF ──(production, BOM b_jl)──► Stock PF    (temps de cycle ct_l)
Stock PF ──(vente)──────────────────► Demande client D_{l,t}
```

---

## Principe de séparation Modèle / Données

> Ce notebook suit le principe **AMPL / GAMS** : le modèle mathématique est défini de façon **purement abstraite** (sections 1 à 4), sans aucune valeur numérique.
> Toutes les données concrètes sont regroupées **en section 5 uniquement**.
> Pour adapter le modèle à des données réelles (SAP), il suffit de modifier la section 5.


---
## 0. Imports

In [ ]:
# Pyomo : librairie de modélisation mathématique (AbstractModel, Var, Param, Constraint, Objective...)
from pyomo.environ import *

# Pandas : manipulation de tableaux de données (affichage des résultats)
import pandas as pd

# NumPy : calculs numériques (formules EOQ, racines carrées)
import numpy as np


---
## 1. LES PARAMÈTRES

### Notations mathématiques

#### Indices (ensembles abstraits)

| Indice | Ensemble | Signification |
|--------|----------|---------------|
| $i$ | $MP$ | Références Matières Premières |
| $j$ | $SF$ | Références Semi-Finies |
| $l$ | $PF$ | Références Produits Finis |
| $t$ | $\{1, \ldots, T\}$ | Périodes mensuelles, $T$ = fin d'horizon fiscal (fin septembre) |

#### Paramètres économiques globaux

| Symbole | Description |
|---------|-------------|
| $Target$ | Cible de valeur de stock totale en fin d'horizon (\$) |
| $\tau$ | Taux de possession annuel (%) |
| $z_{\alpha}$ | Coefficient du niveau de service cible $\alpha$ (ex : $z_{0.95} = 1.65$) |
| $M$ | Grand nombre pour la linéarisation *big-M* (contraintes MOQ / MLS) |
| $V_{max}$ | Capacité de stockage globale (m³) |

#### Paramètres par référence

| Symbole | Étage | Description |
|---------|-------|-------------|
| $p_i^{MP},\, p_j^{SF},\, p_l^{PF}$ | MP / SF / PF | Valeur unitaire (\$/unité) |
| $S_{i,0}^{MP},\, S_{j,0}^{SF},\, S_{l,0}^{PF}$ | tous | Stock initial réel à $t=0$ (extrait SAP) |
| $LT_i$ | MP | Lead time fournisseur (mois) |
| $ct_j,\, ct_l$ | SF / PF | Temps de cycle de production (mois) |
| $MOQ_i$ | MP | Quantité minimale de commande |
| $MLS_j,\, MLS_l$ | SF / PF | Taille de lot minimale de production |
| $c_j,\, c_l$ | SF / PF | Charge unitaire de production (heures/unité) |
| $v_i^{MP},\, v_j^{SF},\, v_l^{PF}$ | tous | Volume unitaire de stockage (m³/unité) |
| $SS_k$ | tous | Stock de sécurité (calculé via EOQ, injecté en data) |

#### Paramètres de flux

| Symbole | Description |
|---------|-------------|
| $D_{l,t}$ | Demande client du PF $l$ en période $t$ (seul étage avec demande externe) |
| $OC_{i,t}^{MP}$ | Commandes fournisseur déjà passées, arrivant en $t$ (figées, extraites SAP) |
| $WIP_{j,t}^{SF},\, WIP_{l,t}^{PF}$ | Encours de production déjà lancés, sortant en $t$ (figés, extraits SAP) |
| $a_{ij}$ | BOM MP $\to$ SF : qté de MP $i$ nécessaire pour 1 unité de SF $j$ |
| $b_{jl}$ | BOM SF $\to$ PF : qté de SF $j$ nécessaire pour 1 unité de PF $l$ |
| $Cap_t^{SF},\, Cap_t^{PF}$ | Capacité de production disponible en $t$ (heures) |

> ⚠️ Aucune valeur numérique dans cette section — uniquement la déclaration des paramètres abstraits Pyomo.


In [ ]:
# Initialisation du modèle abstrait Pyomo
# AbstractModel : le modèle est défini sans données — elles seront injectées via DataPortal (section 5)
model = AbstractModel()

# ── Horizon temporel ─────────────────────────────────────────────────────────
model.T        = Param(within=PositiveIntegers)   # T : dernière période (fin d'horizon fiscal)
model.PER      = RangeSet(1, model.T)             # {1, 2, ..., T} : ensemble des périodes
model.t_juillet = Param(within=PositiveIntegers)  # indice de juillet (dernier mois avant fermeture)
model.AoutSet  = Set(within=model.PER)            # sous-ensemble : période(s) de fermeture fournisseur

# ── Ensembles de références par étage ────────────────────────────────────────
model.MP = Set()   # ensemble des indices Matières Premières
model.SF = Set()   # ensemble des indices Semi-Finis
model.PF = Set()   # ensemble des indices Produits Finis

# ── Paramètres économiques globaux ───────────────────────────────────────────
model.TARGET  = Param()   # cible de valeur de stock fin d'horizon ($)
model.tau     = Param()   # taux de possession annuel (ex : 0.20 = 20 %)
model.z_alpha = Param()   # coefficient niveau de service (ex : 1.65 pour alpha=95%)
model.M       = Param()   # big-M pour linéarisation des contraintes MOQ / MLS
model.V_max   = Param()   # capacité de stockage globale (m³)

# ── Paramètres par référence — Matières Premières ────────────────────────────
model.prix_MP       = Param(model.MP)   # p_i^MP : valeur unitaire ($)
model.stock_init_MP = Param(model.MP)   # S_i,0^MP : stock initial réel (t=0)
model.LT            = Param(model.MP)   # LT_i : lead time fournisseur (mois)
model.MOQ           = Param(model.MP)   # MOQ_i : quantité minimale de commande
model.volume_MP     = Param(model.MP)   # v_i^MP : volume unitaire (m³/unité)
model.SS_MP         = Param(model.MP)   # SS_i : stock de sécurité (pré-calculé EOQ)

# ── Paramètres par référence — Semi-Finis ────────────────────────────────────
model.prix_SF       = Param(model.SF)   # p_j^SF : valeur unitaire ($)
model.stock_init_SF = Param(model.SF)   # S_j,0^SF : stock initial réel (t=0)
model.ct_SF         = Param(model.SF)   # ct_j : temps de cycle production (mois)
model.MLS_SF        = Param(model.SF)   # MLS_j : taille de lot minimale
model.charge_SF     = Param(model.SF)   # c_j : charge unitaire (heures/unité)
model.volume_SF     = Param(model.SF)   # v_j^SF : volume unitaire (m³/unité)
model.SS_SF         = Param(model.SF)   # SS_j : stock de sécurité (pré-calculé EOQ)

# ── Paramètres par référence — Produits Finis ────────────────────────────────
model.prix_PF       = Param(model.PF)   # p_l^PF : valeur unitaire ($)
model.stock_init_PF = Param(model.PF)   # S_l,0^PF : stock initial réel (t=0)
model.ct_PF         = Param(model.PF)   # ct_l : temps de cycle production (mois)
model.MLS_PF        = Param(model.PF)   # MLS_l : taille de lot minimale
model.charge_PF     = Param(model.PF)   # c_l : charge unitaire (heures/unité)
model.volume_PF     = Param(model.PF)   # v_l^PF : volume unitaire (m³/unité)
model.SS_PF         = Param(model.PF)   # SS_l : stock de sécurité (pré-calculé EOQ)

# ── Demande client (uniquement sur les Produits Finis) ───────────────────────
# D_{l,t} : seul l'étage PF a une demande externe réelle (commandes clients)
model.demande = Param(model.PF, model.PER)

# ── Pipeline déjà engagé (données figées, extraites de SAP) ──────────────────
# Ces paramètres représentent des flux DÉJÀ DÉCIDÉS avant le démarrage du modèle.
# Le modèle ne peut pas les modifier — ce sont des constantes.
model.OC     = Param(model.MP, model.PER)   # OC_{i,t} : commandes MP en transit
model.WIP_SF = Param(model.SF, model.PER)   # WIP_{j,t}^SF : encours production SF
model.WIP_PF = Param(model.PF, model.PER)   # WIP_{l,t}^PF : encours production PF

# ── Nomenclature BOM (Bill of Materials) ─────────────────────────────────────
# default=0 : si un couple (i,j) ou (j,l) n'est pas déclaré en data, la valeur est 0
model.a = Param(model.MP, model.SF, default=0)   # a_{ij} : MP -> SF
model.b = Param(model.SF, model.PF, default=0)   # b_{jl} : SF -> PF

# ── Capacités de production par période ──────────────────────────────────────
# Cap_t^SF / Cap_t^PF : exprimées en heures disponibles par mois
# Réduite en août (congés fournisseurs et production)
model.Cap_SF = Param(model.PER)   # capacité atelier SF (heures/mois)
model.Cap_PF = Param(model.PER)   # capacité atelier PF (heures/mois)


---
## 2. LES VARIABLES DE DÉCISION

### Formulation mathématique

Le modèle distingue deux conventions temporelles importantes :

- Les **stocks** $S_{k,t}$ sont des **photos de fin de période** $t$ (après tous les mouvements du mois).
- Les **décisions** ($O$, $P$, $x$, $y$) sont **lancées en début de période** $t$ et produisent leur effet après un délai ($LT_i$ ou $ct_k$).

| Variable | Domaine | Convention | Signification |
|----------|---------|------------|---------------|
| $S_{i,t}^{MP}$ | $\mathbb{R}^+$ | Fin de $t$ | Stock MP $i$ en fin de période $t$ |
| $S_{j,t}^{SF}$ | $\mathbb{R}^+$ | Fin de $t$ | Stock SF $j$ en fin de période $t$ |
| $S_{l,t}^{PF}$ | $\mathbb{R}^+$ | Fin de $t$ | Stock PF $l$ en fin de période $t$ |
| $O_{i,t}^{MP}$ | $\mathbb{R}^+$ | Début de $t$ | Quantité **commandée** (achat MP $i$) lancée en début de $t$ |
| $P_{j,t}^{SF}$ | $\mathbb{R}^+$ | Début de $t$ | Quantité **lancée en production** SF $j$ en début de $t$ |
| $P_{l,t}^{PF}$ | $\mathbb{R}^+$ | Début de $t$ | Quantité **lancée en production** PF $l$ en début de $t$ |
| $x_{i,t}$ | $\{0, 1\}$ | Début de $t$ | **1** si une commande est passée pour MP $i$ en $t$, **0** sinon |
| $y_{j,t}^{SF}$ | $\{0, 1\}$ | Début de $t$ | **1** si un OF est lancé pour SF $j$ en $t$, **0** sinon |
| $y_{l,t}^{PF}$ | $\{0, 1\}$ | Début de $t$ | **1** si un OF est lancé pour PF $l$ en $t$, **0** sinon |

**Note sur le décalage temporel :**
Ce qui est commandé/lancé en début de $t$ n'arrive en stock qu'en fin de $t + LT_i$ (ou $t + ct_k$).
C'est ce décalage qui est encodé dans les contraintes de bilan **C1 / C2 / C3**.


In [ ]:
# ── Stocks en fin de période t ────────────────────────────────────────────────
# domain=NonNegativeReals impose S >= 0 (contrainte C13 implicite)
model.S_MP = Var(model.MP, model.PER, domain=NonNegativeReals)  # stock MP fin de t
model.S_SF = Var(model.SF, model.PER, domain=NonNegativeReals)  # stock SF fin de t
model.S_PF = Var(model.PF, model.PER, domain=NonNegativeReals)  # stock PF fin de t

# ── Quantités commandées (MP) et lancées en production (SF, PF) ───────────────
# Lancées en début de t — l'effet en stock est décalé par LT_i ou ct_k (voir C1/C2/C3)
model.O_MP = Var(model.MP, model.PER, domain=NonNegativeReals)  # O_{i,t}^MP
model.P_SF = Var(model.SF, model.PER, domain=NonNegativeReals)  # P_{j,t}^SF
model.P_PF = Var(model.PF, model.PER, domain=NonNegativeReals)  # P_{l,t}^PF

# ── Variables binaires de déclenchement ──────────────────────────────────────
# domain=Binary impose x, y in {0, 1} (contrainte C13 implicite)
model.x    = Var(model.MP, model.PER, domain=Binary)   # x_{i,t} : commande MP déclenchée ?
model.y_SF = Var(model.SF, model.PER, domain=Binary)   # y_{j,t} : OF SF déclenché ?
model.y_PF = Var(model.PF, model.PER, domain=Binary)   # y_{l,t} : OF PF déclenché ?


---
## 3. LA FONCTION OBJECTIF

### Formulation mathématique

$$\min\; Z = \sum_{t=1}^{T}\left(
  \sum_{i \in MP} p_i^{MP}\, S_{i,t}^{MP}
+ \sum_{j \in SF} p_j^{SF}\, S_{j,t}^{SF}
+ \sum_{l \in PF} p_l^{PF}\, S_{l,t}^{PF}
\right)$$

**Interprétation :**
On minimise la **valeur financière totale du stock immobilisé**, cumulée sur tout l'horizon (juin → septembre).

- Multiplier chaque quantité en stock par son prix unitaire donne la **valeur immobilisée** à cette période.
- Sommer sur toutes les périodes (et pas seulement la dernière) évite un déstockage artificiel concentré en fin d'horizon : le modèle est incité à réduire le stock **dès le début**, pas seulement en septembre.
- La hiérarchie des prix $p^{PF} > p^{SF} > p^{MP}$ fait que le modèle **priorise naturellement la réduction des stocks de produits finis**, qui immobilisent le plus de valeur ajoutée.


In [ ]:
# ── Fonction objectif ─────────────────────────────────────────────────────────
# Minimiser la somme des valeurs de stock (prix * quantité) sur tous les étages et toutes les périodes
def cost_rule(m):
    return (
        # Valeur immobilisée en Matières Premières
        sum(m.prix_MP[i] * m.S_MP[i, t] for i in m.MP for t in m.PER)
        # Valeur immobilisée en Semi-Finis
      + sum(m.prix_SF[j] * m.S_SF[j, t] for j in m.SF for t in m.PER)
        # Valeur immobilisée en Produits Finis
      + sum(m.prix_PF[l] * m.S_PF[l, t] for l in m.PF for t in m.PER)
    )

# sense=minimize : on minimise (par opposition à maximize)
model.cost = Objective(rule=cost_rule, sense=minimize)


---
## 3 (suite). LES CONTRAINTES C1 → C13

---

### C1 — Bilan de stock Matière Première

$$S_{i,t}^{MP} = \underbrace{S_{i,t-1}^{MP}}_{\text{stock précédent}}
+ \underbrace{OC_{i,t}^{MP}}_{\substack{\text{pipeline SAP}\\\text{(figé)}}}
+ \underbrace{O_{i,\;t-LT_i}^{MP}}_{\substack{\text{décision modèle}\\\text{(décalée de }LT_i\text{)}}}
- \underbrace{\sum_{j \in SF} a_{ij}\, P_{j,t}^{SF}}_{\text{consommation par la production SF}}
\quad \forall\, i \in MP,\; t \in T$$

**Points clés :**
- Si $t = 1$ : $S_{i,t-1}^{MP} = S_{i,0}^{MP}$ (stock initial réel SAP) → **C4 intégrée ici**.
- Si $t - LT_i < 1$ : la commande décidée par le modèle n'est pas encore arrivée → terme nul.
- La consommation de MP a lieu **au moment du lancement** de la production SF (pas à sa sortie).

---

### C2 — Bilan de stock Semi-Fini

$$S_{j,t}^{SF} = S_{j,t-1}^{SF}
+ \underbrace{WIP_{j,t}^{SF}}_{\substack{\text{encours déjà}\\\text{lancé (SAP)}}}
+ \underbrace{P_{j,\;t-ct_j}^{SF}}_{\substack{\text{décision modèle}\\\text{(décalée de }ct_j\text{)}}}
- \underbrace{\sum_{l \in PF} b_{jl}\, P_{l,t}^{PF}}_{\text{consommation par la production PF}}
\quad \forall\, j \in SF,\; t \in T$$

---

### C3 — Bilan de stock Produit Fini

$$S_{l,t}^{PF} = S_{l,t-1}^{PF}
+ \underbrace{WIP_{l,t}^{PF}}_{\text{encours (SAP)}}
+ \underbrace{P_{l,\;t-ct_l}^{PF}}_{\text{décision modèle}}
- \underbrace{D_{l,t}}_{\substack{\text{demande client}\\\text{(seul étage avec}\\\text{demande externe)}}}
\quad \forall\, l \in PF,\; t \in T$$

---

### C4 — Initialisation des stocks

$$S_{i,0}^{MP} = \text{stock\_init\_MP}[i], \quad
S_{j,0}^{SF} = \text{stock\_init\_SF}[j], \quad
S_{l,0}^{PF} = \text{stock\_init\_PF}[l]$$

> **Intégrée dans C1/C2/C3** via la condition $t = 1$ — pas de contrainte Pyomo séparée.

---

### C5 — Cible de valeur de stock fin d'année fiscale

$$\sum_{i \in MP} p_i^{MP}\, S_{i,T}^{MP}
+ \sum_{j \in SF} p_j^{SF}\, S_{j,T}^{SF}
+ \sum_{l \in PF} p_l^{PF}\, S_{l,T}^{PF}
\leq Target$$

**Contrainte principale du modèle** : la valeur totale du stock en fin d'horizon (fin septembre) doit être inférieure ou égale à la cible fiscale $Target$.

---

### C6 — Taux de service minimum (Produit Fini uniquement)

$$S_{l,t}^{PF} \geq SS_l \quad \forall\, l \in PF,\; t \in T$$

Le stock de PF ne doit jamais descendre en dessous du stock de sécurité $SS_l$ (calculé via EOQ), garantissant un niveau de service $\alpha$ vis-à-vis des clients.

---

### C7 — Capacité de production

$$\underbrace{\sum_{j \in SF} c_j\, P_{j,t}^{SF}}_{\text{charge totale générée}} \leq Cap_t^{SF} \quad \forall\, t$$

$$\underbrace{\sum_{l \in PF} c_l\, P_{l,t}^{PF}}_{\text{charge totale générée}} \leq Cap_t^{PF} \quad \forall\, t$$

- $c_k$ = **charge unitaire** (heures nécessaires pour produire 1 unité) — paramètre par référence.
- $Cap_t$ = **capacité disponible** (heures totales du mois) — réduite en août pour les congés.

---

### C8 — Capacité de stockage globale (volume physique)

$$\sum_{i \in MP} v_i^{MP}\, S_{i,t}^{MP}
+ \sum_{j \in SF} v_j^{SF}\, S_{j,t}^{SF}
+ \sum_{l \in PF} v_l^{PF}\, S_{l,t}^{PF}
\leq V_{max} \quad \forall\, t$$

Le volume total occupé ne doit pas dépasser la capacité physique de l'entrepôt.

---

### C9 — MOQ et liaison binaire — Achat Matière Première

$$O_{i,t}^{MP} \geq MOQ_i \cdot x_{i,t} \quad \forall\, i, t \qquad \text{(quantité min si commande)}$$
$$O_{i,t}^{MP} \leq M \cdot x_{i,t} \quad \forall\, i, t \qquad \text{(force zéro si pas de commande)}$$

- Si $x_{i,t} = 0$ → $O_{i,t}^{MP} = 0$ (aucune commande)
- Si $x_{i,t} = 1$ → $O_{i,t}^{MP} \geq MOQ_i$ (commande ≥ quantité minimale)

---

### C10 — MLS et liaison binaire — Production SF et PF

$$P_{j,t}^{SF} \geq MLS_j \cdot y_{j,t}^{SF}, \quad P_{j,t}^{SF} \leq M \cdot y_{j,t}^{SF} \quad \forall\, j, t$$
$$P_{l,t}^{PF} \geq MLS_l \cdot y_{l,t}^{PF}, \quad P_{l,t}^{PF} \leq M \cdot y_{l,t}^{PF} \quad \forall\, l, t$$

Même logique que C9 : si un OF est lancé ($y = 1$), la quantité produite doit respecter la taille de lot minimale $MLS$.

---

### C11 — Fermeture fournisseur en août

$$O_{i,t}^{MP} = 0 \quad \forall\, i \in MP,\; t \in t_{ao\hat{u}t}$$

Les fournisseurs sont fermés en août : aucune nouvelle commande MP ne peut être passée.

---

### C12 — Couverture de stock avant fermeture

$$S_{i,\, t_{juillet}}^{MP} \geq \sum_{t \in t_{ao\hat{u}t}} \sum_{j \in SF} a_{ij}\, P_{j,t}^{SF} \quad \forall\, i \in MP$$

Le stock MP disponible fin juillet doit couvrir **toute la consommation prévue en août**, puisqu'aucun réapprovisionnement ne sera possible.
Cette contrainte force le modèle à constituer un **stock tampon en juillet** → pic visible sur la trajectoire.

---

### C13 — Non-négativité et domaine des variables

$$S^{MP},\, S^{SF},\, S^{PF},\, O^{MP},\, P^{SF},\, P^{PF} \geq 0 \qquad x_{i,t},\, y_{j,t}^{SF},\, y_{l,t}^{PF} \in \{0, 1\}$$

> **Intégrée dans la déclaration des variables** via `domain=NonNegativeReals` et `domain=Binary` — pas de contrainte Pyomo séparée.


In [ ]:
# ── C1 — Bilan de stock Matière Première ─────────────────────────────────────
# S[i,t] = S[i,t-1] + OC[i,t] + O[i, t-LT_i] - sum_j(a[i,j] * P_SF[j,t])
def c1_rule(m, i, t):
    # Stock de la période précédente (ou stock initial si t=1 → intègre C4)
    S_prev = m.stock_init_MP[i] if t == 1 else m.S_MP[i, t-1]

    # Réception issue d'une commande DÉCIDÉE par le modèle, lancée t-LT_i périodes avant
    t_cmd     = t - m.LT[i]
    reception = m.O_MP[i, t_cmd] if t_cmd >= 1 else 0  # nul si hors horizon

    # Consommation de MP i par toutes les productions SF lancées en t (via BOM)
    conso = sum(m.a[i, j] * m.P_SF[j, t] for j in m.SF if m.a[i, j] > 0)

    return m.S_MP[i, t] == S_prev + m.OC[i, t] + reception - conso

model.C1_Bilan_MP = Constraint(model.MP, model.PER, rule=c1_rule)

# ── C2 — Bilan de stock Semi-Fini ─────────────────────────────────────────────
# S[j,t] = S[j,t-1] + WIP_SF[j,t] + P_SF[j, t-ct_j] - sum_l(b[j,l] * P_PF[l,t])
def c2_rule(m, j, t):
    S_prev = m.stock_init_SF[j] if t == 1 else m.S_SF[j, t-1]

    # Sortie de production SF décidée par le modèle, lancée t-ct_j périodes avant
    t_lanc    = t - m.ct_SF[j]
    reception = m.P_SF[j, t_lanc] if t_lanc >= 1 else 0

    # Consommation de SF j par toutes les productions PF lancées en t (via BOM)
    conso = sum(m.b[j, l] * m.P_PF[l, t] for l in m.PF if m.b[j, l] > 0)

    return m.S_SF[j, t] == S_prev + m.WIP_SF[j, t] + reception - conso

model.C2_Bilan_SF = Constraint(model.SF, model.PER, rule=c2_rule)

# ── C3 — Bilan de stock Produit Fini ──────────────────────────────────────────
# S[l,t] = S[l,t-1] + WIP_PF[l,t] + P_PF[l, t-ct_l] - D[l,t]
def c3_rule(m, l, t):
    S_prev = m.stock_init_PF[l] if t == 1 else m.S_PF[l, t-1]

    # Sortie de production PF décidée par le modèle, lancée t-ct_l périodes avant
    t_lanc    = t - m.ct_PF[l]
    reception = m.P_PF[l, t_lanc] if t_lanc >= 1 else 0

    # Demande client externe : seul flux de sortie non contrôlable par le modèle
    return m.S_PF[l, t] == S_prev + m.WIP_PF[l, t] + reception - m.demande[l, t]

model.C3_Bilan_PF = Constraint(model.PF, model.PER, rule=c3_rule)

# C4 — Initialisation : intégrée dans C1/C2/C3 via la condition t==1 (pas de contrainte séparée)

# ── C5 — Cible de valeur de stock fin d'année fiscale ─────────────────────────
# sum_i(p_i * S_MP[i,T]) + sum_j(p_j * S_SF[j,T]) + sum_l(p_l * S_PF[l,T]) <= TARGET
def c5_rule(m):
    return (
        sum(m.prix_MP[i] * m.S_MP[i, m.T] for i in m.MP)
      + sum(m.prix_SF[j] * m.S_SF[j, m.T] for j in m.SF)
      + sum(m.prix_PF[l] * m.S_PF[l, m.T] for l in m.PF)
      <= m.TARGET
    )
model.C5_Cible_Fiscale = Constraint(rule=c5_rule)

# ── C6 — Taux de service minimum (Produits Finis uniquement) ──────────────────
# S_PF[l,t] >= SS_l pour tout l, t
def c6_rule(m, l, t):
    return m.S_PF[l, t] >= m.SS_PF[l]   # SS_PF[l] calculé via EOQ, injecté en data
model.C6_TauxService = Constraint(model.PF, model.PER, rule=c6_rule)

# ── C7 — Capacité de production (charge totale <= capacité disponible) ─────────
# sum_j(c_j * P_SF[j,t]) <= Cap_SF[t]   pour tout t
def c7_sf_rule(m, t):
    return sum(m.charge_SF[j] * m.P_SF[j, t] for j in m.SF) <= m.Cap_SF[t]
model.C7_Capacite_SF = Constraint(model.PER, rule=c7_sf_rule)

# sum_l(c_l * P_PF[l,t]) <= Cap_PF[t]   pour tout t
def c7_pf_rule(m, t):
    return sum(m.charge_PF[l] * m.P_PF[l, t] for l in m.PF) <= m.Cap_PF[t]
model.C7_Capacite_PF = Constraint(model.PER, rule=c7_pf_rule)

# ── C8 — Capacité de stockage globale (volume physique) ───────────────────────
# sum(v_i*S_MP + v_j*S_SF + v_l*S_PF) <= V_max   pour tout t
def c8_rule(m, t):
    return (
        sum(m.volume_MP[i] * m.S_MP[i, t] for i in m.MP)
      + sum(m.volume_SF[j] * m.S_SF[j, t] for j in m.SF)
      + sum(m.volume_PF[l] * m.S_PF[l, t] for l in m.PF)
      <= m.V_max
    )
model.C8_Capacite_Stockage = Constraint(model.PER, rule=c8_rule)

# ── C9 — MOQ et liaison binaire — Achat MP ────────────────────────────────────
# O_MP[i,t] >= MOQ_i * x[i,t]   : si commande, quantité >= MOQ
def c9a_rule(m, i, t):
    return m.O_MP[i, t] >= m.MOQ[i] * m.x[i, t]
model.C9a_MOQ_min = Constraint(model.MP, model.PER, rule=c9a_rule)

# O_MP[i,t] <= M * x[i,t]   : si pas de commande (x=0), force O_MP=0
def c9b_rule(m, i, t):
    return m.O_MP[i, t] <= m.M * m.x[i, t]
model.C9b_MOQ_max = Constraint(model.MP, model.PER, rule=c9b_rule)

# ── C10 — MLS et liaison binaire — Production SF / PF ─────────────────────────
# Même logique que C9 mais pour les lancements de production

# P_SF[j,t] >= MLS_j * y_SF[j,t]
def c10a_sf_rule(m, j, t):
    return m.P_SF[j, t] >= m.MLS_SF[j] * m.y_SF[j, t]
model.C10a_MLS_SF_min = Constraint(model.SF, model.PER, rule=c10a_sf_rule)

# P_SF[j,t] <= M * y_SF[j,t]
def c10b_sf_rule(m, j, t):
    return m.P_SF[j, t] <= m.M * m.y_SF[j, t]
model.C10b_MLS_SF_max = Constraint(model.SF, model.PER, rule=c10b_sf_rule)

# P_PF[l,t] >= MLS_l * y_PF[l,t]
def c10a_pf_rule(m, l, t):
    return m.P_PF[l, t] >= m.MLS_PF[l] * m.y_PF[l, t]
model.C10a_MLS_PF_min = Constraint(model.PF, model.PER, rule=c10a_pf_rule)

# P_PF[l,t] <= M * y_PF[l,t]
def c10b_pf_rule(m, l, t):
    return m.P_PF[l, t] <= m.M * m.y_PF[l, t]
model.C10b_MLS_PF_max = Constraint(model.PF, model.PER, rule=c10b_pf_rule)

# ── C11 — Fermeture fournisseur en août (aucune commande possible) ─────────────
# O_MP[i,t] = 0  pour tout i, t dans AoutSet
def c11_rule(m, i, t):
    return m.O_MP[i, t] == 0
model.C11_Fermeture_Fournisseur = Constraint(model.MP, model.AoutSet, rule=c11_rule)

# ── C12 — Couverture de stock avant fermeture ──────────────────────────────────
# S_MP[i, t_juillet] >= sum_{t in AoutSet} sum_j(a[i,j] * P_SF[j,t])
# Stock fin juillet >= toute la consommation MP prévue en août
def c12_rule(m, i):
    conso_aout = sum(
        m.a[i, j] * m.P_SF[j, t]
        for t in m.AoutSet
        for j in m.SF
        if m.a[i, j] > 0
    )
    return m.S_MP[i, m.t_juillet] >= conso_aout
model.C12_Couverture_Aout = Constraint(model.MP, rule=c12_rule)

# C13 — Non-négativité et domaine binaire :
#        déjà intégrés via domain=NonNegativeReals et domain=Binary dans la déclaration des variables


---
## 4. RÉSOLUTION (fonction générique)

Cette fonction encapsule l'appel au solveur.
Elle sera appelée dans la section 5 (données), après la construction de l'instance concrète.


In [ ]:
def resoudre(instance, solveur="cbc", verbose=True):
    """
    Résout l'instance concrète du modèle MILP.

    Paramètres
    ----------
    instance : instance Pyomo concrète (créée via model.create_instance(data))
    solveur  : nom du solveur à utiliser (défaut : 'cbc', open-source)
    verbose  : afficher les logs du solveur (défaut : True)

    Retourne
    --------
    resultats : objet Pyomo contenant le statut et les informations de résolution
    """
    opt       = SolverFactory(solveur)   # initialise le solveur
    resultats = opt.solve(instance, tee=verbose)
    return resultats


---
### 5.1 Import des données réelles (Excel) + génération logique des données manquantes

> **Ce qui change par rapport à la version « données fictives » :** seules les cellules 5.1 et
> 5.2 changent. Les sections 1 à 4 (modèle abstrait) et 5.3 à 5.8 (DataPortal, résolution,
> affichage, graphiques) restent **strictement identiques** — elles consomment toujours les
> mêmes variables Python (`donnees_MP`, `donnees_SF`, `donnees_PF`, `a_bom`, `b_bom`, `OC`,
> `WIP_SF`, `WIP_PF`, `volume_MP/SF/PF`, `Cap_SF`, `Cap_PF`, `TARGET_val`, `V_max_val`), seule
> leur **source** change (Excel au lieu de dictionnaires codés en dur).

**Fichiers attendus (adapte les noms de colonnes ci-dessous à tes fichiers réels) :**

| Fichier | Colonnes attendues | Utilisé pour |
|---|---|---|
| `produits.xlsx` | `PN`, `Type` (MP/SF/PF), `Stock_Actuel`, `Prix_Unitaire` | stock initial, prix, classification par étage |
| `demande.xlsx` | `PN`, puis une colonne par semaine (ex. `FY25-S23`) | demande client (PF uniquement) |
| `leadtimes.xlsx` | `PN`, `Fournisseur`, `LT_jours` | lead time fournisseur MP (converti en mois) |
| `wip_achats.xlsx` | `PN`, `Quantite` | encours production (WIP) + achats déjà engagés (OC) |
| `moq_mls.xlsx` | `PN`, `Valeur` | MOQ (MP) ou MLS (SF/PF) selon le type du PN |
| `bom_mp_sf.xlsx` / `bom_sf_pf.xlsx` | *(à venir demain, extraction SAP)* | nomenclature — **placeholder ci-dessous en attendant** |

**Générées de façon logique (pas de fichier source), documentées dans le code :**
`volume_MP/SF/PF` (m³/unité), `charge_SF/PF` (h/unité), `Cap_SF/Cap_PF` (h/mois), `C_passation`,
`C_lancement`, `ct_SF/PF` — chacune avec l'hypothèse métier utilisée en commentaire, à ajuster
dès que tu as les vraies valeurs.


In [ ]:
import math
import re
from datetime import date

# ============================================================
# 0) FICHIERS SOURCES — adapte les chemins à tes fichiers réels
# ============================================================
FICHIER_PRODUITS   = "produits.xlsx"     # PN, Type (MP/SF/PF), Stock_Actuel, Prix_Unitaire
FICHIER_DEMANDE    = "demande.xlsx"      # PN, puis une colonne par semaine (ex: "FY25-S23")
FICHIER_LEADTIMES  = "leadtimes.xlsx"    # PN, Fournisseur, LT_jours
FICHIER_WIP_ACHATS = "wip_achats.xlsx"   # PN, Quantite (encours + achats déjà engagés)
FICHIER_MOQ_MLS    = "moq_mls.xlsx"      # PN, Valeur (MOQ si MP, MLS si SF/PF)

# ============================================================
# 1) HORIZON & CALENDRIER — ⚠️ adapte si ton horizon fiscal diffère
# ============================================================
noms_periodes = {1: "Juin", 2: "Juillet", 3: "Août", 4: "Septembre"}
mois_vers_t   = {6: 1, 7: 2, 8: 3, 9: 4}

# ============================================================
# 2) RÉFÉRENTIEL PRODUITS (type, stock actuel, prix)
# ============================================================
df_produits = pd.read_excel(FICHIER_PRODUITS).set_index("PN")

refs_MP = df_produits[df_produits["Type"] == "MP"].index.tolist()
refs_SF = df_produits[df_produits["Type"] == "SF"].index.tolist()
refs_PF = df_produits[df_produits["Type"] == "PF"].index.tolist()

# ============================================================
# 3) LEAD TIMES (MP uniquement) — conversion jours -> mois
# ============================================================
df_lt = pd.read_excel(FICHIER_LEADTIMES).groupby("PN")["LT_jours"].max()  # max si multi-fournisseurs
LT_mois = {i: max(1, math.ceil(df_lt.get(i, 30) / 30)) for i in refs_MP}

# ============================================================
# 4) MOQ (MP) / MLS (SF, PF)
# ============================================================
df_moqmls = pd.read_excel(FICHIER_MOQ_MLS).set_index("PN")["Valeur"]
MOQ    = {i: df_moqmls.get(i, 1) for i in refs_MP}
MLS_SF = {j: df_moqmls.get(j, 1) for j in refs_SF}
MLS_PF = {l: df_moqmls.get(l, 1) for l in refs_PF}

# ============================================================
# 5) WIP / ACHATS DÉJÀ ENGAGÉS — tout affecté à la période 1
#    ⚠️ si ton fichier a une colonne de date/période d'arrivée, remplace ce bloc
#    par une vraie répartition par t au lieu de tout mettre en t=1
# ============================================================
df_wip = pd.read_excel(FICHIER_WIP_ACHATS).set_index("PN")["Quantite"]
OC     = {i: {1: df_wip.get(i, 0), 2: 0, 3: 0, 4: 0} for i in refs_MP}
WIP_SF = {j: {1: df_wip.get(j, 0), 2: 0, 3: 0, 4: 0} for j in refs_SF}
WIP_PF = {l: {1: df_wip.get(l, 0), 2: 0, 3: 0, 4: 0} for l in refs_PF}

# ============================================================
# 6) DEMANDE (PF uniquement) — passage semaine -> mois
# ============================================================
df_dem = pd.read_excel(FICHIER_DEMANDE).set_index("PN")
cols_semaines = list(df_dem.columns)

def semaine_vers_mois(nom_colonne):
    """⚠️ À ADAPTER : suppose un format 'FYaa-Sww' (ex: 'FY25-S23')."""
    m = re.search(r"FY(\d+).*?S(\d+)", str(nom_colonne))
    if not m:
        return None
    annee, semaine = 2000 + int(m.group(1)), int(m.group(2))
    try:
        return date.fromisocalendar(annee, semaine, 1).month
    except ValueError:
        return None

demande_mensuelle    = {l: {t: 0 for t in range(1, 5)} for l in refs_PF}
demande_hebdo_liste  = {l: [] for l in refs_PF}   # conservé pour calculer un sigma_D réel

for col in cols_semaines:
    mois = semaine_vers_mois(col)
    if mois not in mois_vers_t:
        continue
    t = mois_vers_t[mois]
    for l in refs_PF:
        qte = df_dem.loc[l, col] if l in df_dem.index else 0
        demande_mensuelle[l][t] += qte
        demande_hebdo_liste[l].append(qte)

# ============================================================
# 7) sigma_D — écart-type de la demande CALCULÉ RÉELLEMENT (PF)
#    Pour MP/SF, en attendant la BOM (demain), on propage la moyenne des sigma_D PF
#    (proxy temporaire à affiner une fois la nomenclature connue)
# ============================================================
sigma_D_PF    = {l: (np.std(demande_hebdo_liste[l]) if demande_hebdo_liste[l] else 1.0) for l in refs_PF}
sigma_D_proxy = float(np.mean(list(sigma_D_PF.values()))) if sigma_D_PF else 10.0
sigma_D_MP    = {i: sigma_D_proxy for i in refs_MP}
sigma_D_SF    = {j: sigma_D_proxy for j in refs_SF}

# ============================================================
# 8) GÉNÉRATION LOGIQUE — coûts fixes de passation / lancement
#    Hypothèse : coût fixe ≈ 0.5x le prix unitaire, borné dans une fourchette réaliste
#    (à remplacer par les vrais coûts achat/ordonnancement dès que disponibles)
# ============================================================
def cout_fixe_logique(prix, plancher, plafond):
    return float(np.clip(prix * 0.5, plancher, plafond))

C_passation    = {i: cout_fixe_logique(df_produits.loc[i, "Prix_Unitaire"], 50, 1000)  for i in refs_MP}
C_lancement_SF = {j: cout_fixe_logique(df_produits.loc[j, "Prix_Unitaire"], 100, 1500) for j in refs_SF}
C_lancement_PF = {l: cout_fixe_logique(df_produits.loc[l, "Prix_Unitaire"], 150, 2000) for l in refs_PF}

# ============================================================
# 9) GÉNÉRATION LOGIQUE — volumes unitaires de stockage (m³/unité)
#    Hypothèse : volume de base croissant MP < SF < PF (assemblage), mais un produit plus
#    cher est généralement plus compact/precis (électronique) -> volume décroît avec le prix
# ============================================================
def volume_logique(prix, etage):
    base = {"MP": 0.02, "SF": 0.05, "PF": 0.12}[etage]
    return round(base / (1 + prix / 500), 4)

volume_MP = {i: volume_logique(df_produits.loc[i, "Prix_Unitaire"], "MP") for i in refs_MP}
volume_SF = {j: volume_logique(df_produits.loc[j, "Prix_Unitaire"], "SF") for j in refs_SF}
volume_PF = {l: volume_logique(df_produits.loc[l, "Prix_Unitaire"], "PF") for l in refs_PF}

# ============================================================
# 10) GÉNÉRATION LOGIQUE — charge de production (h/unité), SF & PF uniquement
#     Hypothèse : charge ≈ prix / 150, bornée dans une fourchette réaliste de petite série
# ============================================================
def charge_logique(prix, plancher, plafond):
    return float(np.clip(prix / 150, plancher, plafond))

charge_SF = {j: charge_logique(df_produits.loc[j, "Prix_Unitaire"], 0.5, 8.0)  for j in refs_SF}
charge_PF = {l: charge_logique(df_produits.loc[l, "Prix_Unitaire"], 1.0, 12.0) for l in refs_PF}

# ============================================================
# 11) GÉNÉRATION LOGIQUE — temps de cycle production ct (mois)
#     Hypothèse : cycle long (2 mois) pour le quartile de charge le plus élevé, sinon 1 mois
# ============================================================
seuil_SF = np.percentile(list(charge_SF.values()), 75) if charge_SF else 0
seuil_PF = np.percentile(list(charge_PF.values()), 75) if charge_PF else 0
ct_SF = {j: (2 if charge_SF[j] > seuil_SF else 1) for j in refs_SF}
ct_PF = {l: (2 if charge_PF[l] > seuil_PF else 1) for l in refs_PF}

# ============================================================
# 12) ASSEMBLAGE FINAL — mêmes dictionnaires que la version « données fictives »
# ============================================================
donnees_MP = {
    i: {"prix": df_produits.loc[i, "Prix_Unitaire"], "stock_init": df_produits.loc[i, "Stock_Actuel"],
        "LT": LT_mois[i], "MOQ": MOQ[i], "sigma_D": sigma_D_MP[i], "C_passation": C_passation[i]}
    for i in refs_MP
}
donnees_SF = {
    j: {"prix": df_produits.loc[j, "Prix_Unitaire"], "stock_init": df_produits.loc[j, "Stock_Actuel"],
        "ct": ct_SF[j], "MLS": MLS_SF[j], "sigma_D": sigma_D_SF[j],
        "C_lancement": C_lancement_SF[j], "charge": charge_SF[j]}
    for j in refs_SF
}
donnees_PF = {
    l: {"prix": df_produits.loc[l, "Prix_Unitaire"], "stock_init": df_produits.loc[l, "Stock_Actuel"],
        "ct": ct_PF[l], "MLS": MLS_PF[l], "demande": demande_mensuelle[l],
        "sigma_D": sigma_D_PF[l], "C_lancement": C_lancement_PF[l], "charge": charge_PF[l]}
    for l in refs_PF
}

# ============================================================
# 13) BOM — PLACEHOLDER en attendant l'extraction SAP (demain)
#     ⚠️ À REMPLACER dès réception, par exemple :
#       df_a = pd.read_excel("bom_mp_sf.xlsx")   # colonnes: MP, SF, Quantite
#       a_bom = {i: {} for i in refs_MP}
#       for _, r in df_a.iterrows(): a_bom[r["MP"]][r["SF"]] = r["Quantite"]
#       (idem pour b_bom avec bom_sf_pf.xlsx : colonnes SF, PF, Quantite)
# ============================================================
a_bom = {i: {j: 0 for j in refs_SF} for i in refs_MP}
b_bom = {j: {l: 0 for l in refs_PF} for j in refs_SF}
print("⚠️  BOM PLACEHOLDER (tout à 0) — à remplacer dès réception de l'extraction SAP demain")

# ============================================================
# 14) CAPACITÉS DE PRODUCTION — calcul réaliste via calendrier ouvré
#     Hypothèse : 2 lignes de production, 7.5h/jour/ligne, congés d'été concentrés en août
#     ⚠️ Ajuste jours_ouvres / heures_jour / nb_lignes à la réalité de ton site
# ============================================================
jours_ouvres = {1: 21, 2: 22, 3: 10, 4: 21}   # Juin, Juillet, Août (congés), Septembre
heures_jour, nb_lignes = 7.5, 2

Cap_SF = {t: jours_ouvres[t] * heures_jour * nb_lignes for t in range(1, 5)}
Cap_PF = {t: jours_ouvres[t] * heures_jour * nb_lignes for t in range(1, 5)}

# ============================================================
# 15) PARAMÈTRES GLOBAUX — ⚠️ à ajuster aux vraies valeurs du site
# ============================================================
TARGET_val = 10_000_000
V_max_val  = 5000

# ============================================================
# 16) SYNTHÈSE
# ============================================================
print("=" * 60)
print("  DONNÉES RÉELLES CHARGÉES + DONNÉES GÉNÉRÉES — SYNTHÈSE")
print("=" * 60)
print(f"  Références MP : {refs_MP}")
print(f"  Références SF : {refs_SF}")
print(f"  Références PF : {refs_PF}")
print()

val_MP_init    = sum(p["prix"] * p["stock_init"] for p in donnees_MP.values())
val_SF_init    = sum(p["prix"] * p["stock_init"] for p in donnees_SF.values())
val_PF_init    = sum(p["prix"] * p["stock_init"] for p in donnees_PF.values())
val_total_init = val_MP_init + val_SF_init + val_PF_init

print("  Valeur initiale du stock (t=0) :")
print(f"    MP  : {val_MP_init:>15,.0f} $")
print(f"    SF  : {val_SF_init:>15,.0f} $")
print(f"    PF  : {val_PF_init:>15,.0f} $")
print(f"    TOTAL : {val_total_init:>13,.0f} $")
print(f"  Cible fiscale (TARGET) : {TARGET_val:>14,.0f} $")
print(f"  Écart à réduire        : {val_total_init - TARGET_val:>14,.0f} $")
print("=" * 60)

df_cap = pd.DataFrame({"Cap SF (h)": Cap_SF, "Cap PF (h)": Cap_PF})
df_cap.index = [noms_periodes[t] for t in df_cap.index]
df_cap.index.name = "Période"
display(df_cap)


---
### 5.1bis — Export des données générées (pour réutilisation future)

Les données générées de façon logique à l'étape précédente (volumes, charge, cycle de
production, coûts fixes, capacités) sont exportées ici en fichiers Excel. La prochaine fois,
tu pourras **les rouvrir, les corriger/valider avec les vraies valeurs**, puis les réimporter
directement comme n'importe lequel de tes fichiers sources (au lieu de les régénérer).


In [ ]:
import os
outdir = "donnees_generees"   # ⚠️ change le dossier de sortie si besoin
os.makedirs(outdir, exist_ok=True)

# --- volumes.xlsx ---
df_vol = pd.DataFrame(
    [{"PN": i, "Type": "MP", "Volume_Unitaire_m3": volume_MP[i]} for i in refs_MP] +
    [{"PN": j, "Type": "SF", "Volume_Unitaire_m3": volume_SF[j]} for j in refs_SF] +
    [{"PN": l, "Type": "PF", "Volume_Unitaire_m3": volume_PF[l]} for l in refs_PF]
)
df_vol.to_excel(f"{outdir}/volumes.xlsx", index=False)

# --- charge_production.xlsx (SF/PF uniquement, + cycle ct) ---
df_charge = pd.DataFrame(
    [{"PN": j, "Type": "SF", "Charge_Unitaire_h": charge_SF[j], "Ct_Mois": ct_SF[j]} for j in refs_SF] +
    [{"PN": l, "Type": "PF", "Charge_Unitaire_h": charge_PF[l], "Ct_Mois": ct_PF[l]} for l in refs_PF]
)
df_charge.to_excel(f"{outdir}/charge_production.xlsx", index=False)

# --- couts_fixes.xlsx (passation MP / lancement SF-PF) ---
df_couts = pd.DataFrame(
    [{"PN": i, "Type": "MP", "Cout_Fixe": C_passation[i]}    for i in refs_MP] +
    [{"PN": j, "Type": "SF", "Cout_Fixe": C_lancement_SF[j]} for j in refs_SF] +
    [{"PN": l, "Type": "PF", "Cout_Fixe": C_lancement_PF[l]} for l in refs_PF]
)
df_couts.to_excel(f"{outdir}/couts_fixes.xlsx", index=False)

# --- capacites_production.xlsx ---
df_cap_export = pd.DataFrame({
    "t": list(range(1, 5)),
    "Periode": [noms_periodes[t] for t in range(1, 5)],
    "Cap_SF_h": [Cap_SF[t] for t in range(1, 5)],
    "Cap_PF_h": [Cap_PF[t] for t in range(1, 5)],
})
df_cap_export.to_excel(f"{outdir}/capacites_production.xlsx", index=False)

# --- parametres_globaux.xlsx ---
df_params = pd.DataFrame([
    {"Parametre": "TARGET",  "Valeur": TARGET_val},
    {"Parametre": "V_max",   "Valeur": V_max_val},
    {"Parametre": "tau",     "Valeur": tau},
    {"Parametre": "z_alpha", "Valeur": z_alpha},
])
df_params.to_excel(f"{outdir}/parametres_globaux.xlsx", index=False)

print(f"✅ Fichiers exportés dans '{outdir}/' :")
for f in ["volumes.xlsx", "charge_production.xlsx", "couts_fixes.xlsx",
          "capacites_production.xlsx", "parametres_globaux.xlsx"]:
    print(" -", f)


> **La prochaine fois**, remplace simplement les blocs « GÉNÉRATION LOGIQUE » (9, 10, 11, 8, 14, 15)
> de la cellule 5.1 par une lecture de ces fichiers, par exemple :
> ```python
> df_vol = pd.read_excel("donnees_generees/volumes.xlsx").set_index("PN")["Volume_Unitaire_m3"]
> volume_MP = {i: df_vol[i] for i in refs_MP}
> ```
> Tout le reste du notebook (5.2 à 5.8) reste identique.


---
### 5.2 Calibration EOQ — calcul des stocks de sécurité `SS`

**Formules :**

$$Q_k^* = \sqrt{\frac{2\,D_k\,C_k}{h_k}}, \qquad h_k = \tau \cdot p_k, \qquad SS_k = z_{\alpha}\,\sigma_{D_k}\,\sqrt{LT_k \text{ ou } ct_k}$$

Les valeurs $SS_k$ calculées ici sont injectées dans le `DataPortal` (section 5.3) et utilisées par la contrainte **C6**.

In [ ]:
tau, z_alpha = 0.20, 1.65   # paramètres économiques (taux possession, niveau service)

# Demandes annuelles approchées pour le calcul EOQ
# ⚠️ proxy temporaire (stock_init x 4) en attendant la BOM (demain) pour un vrai calcul
# par cascade de la demande PF réelle à travers la nomenclature
D_annuel_MP = {i: donnees_MP[i]["stock_init"] * 4 for i in donnees_MP}
D_annuel_SF = {j: donnees_SF[j]["stock_init"] * 4 for j in donnees_SF}

SS_MP, SS_SF, SS_PF = {}, {}, {}
resultats_eoq = []

# ── MP ────────────────────────────────────────────────────────────────────────
for i, p in donnees_MP.items():
    h      = tau * p["prix"]                                    # coût de possession unitaire
    Q_star = np.sqrt(2 * D_annuel_MP[i] * p["C_passation"] / h)  # quantité économique
    ss     = z_alpha * p["sigma_D"] * np.sqrt(p["LT"])           # stock de sécurité
    SS_MP[i] = ss
    resultats_eoq.append({"Référence": i, "Étage": "MP", "h ($/an)": round(h,2),
                           "EOQ* (unités)": round(Q_star), "SS (unités)": round(ss)})

# ── SF ────────────────────────────────────────────────────────────────────────
for j, p in donnees_SF.items():
    h      = tau * p["prix"]
    Q_star = np.sqrt(2 * D_annuel_SF[j] * p["C_lancement"] / h)
    ss     = z_alpha * p["sigma_D"] * np.sqrt(p["ct"])
    SS_SF[j] = ss
    resultats_eoq.append({"Référence": j, "Étage": "SF", "h ($/an)": round(h,2),
                           "EOQ* (unités)": round(Q_star), "SS (unités)": round(ss)})

# ── PF ────────────────────────────────────────────────────────────────────────
for l, p in donnees_PF.items():
    h      = tau * p["prix"]
    D_an   = sum(p["demande"].values()) * 3   # annualisé (4 mois * 3)
    Q_star = np.sqrt(2 * D_an * p["C_lancement"] / h)
    ss     = z_alpha * p["sigma_D"] * np.sqrt(p["ct"])
    SS_PF[l] = ss
    resultats_eoq.append({"Référence": l, "Étage": "PF", "h ($/an)": round(h,2),
                           "EOQ* (unités)": round(Q_star), "SS (unités)": round(ss)})

print("Calibration EOQ — Stocks de sécurité calculés :")
display(pd.DataFrame(resultats_eoq).set_index("Référence"))


---
### 5.3 Construction du `DataPortal` et de l'instance concrète

Le `DataPortal` est le mécanisme Pyomo qui injecte les valeurs concrètes dans le modèle abstrait.
Chaque clé correspond à un paramètre ou ensemble déclaré en section 1.

In [ ]:
data = DataPortal()

# ── Horizon temporel ─────────────────────────────────────────────────────────
data['T']         = 4     # T_max : fin septembre
data['t_juillet'] = 2     # index juillet
data['AoutSet']   = [3]   # index(s) du/des mois de fermeture fournisseur

# ── Ensembles de références ───────────────────────────────────────────────────
data['MP'] = list(donnees_MP.keys())
data['SF'] = list(donnees_SF.keys())
data['PF'] = list(donnees_PF.keys())

# ── Paramètres globaux ────────────────────────────────────────────────────────
data['TARGET']  = TARGET_val    # cible fiscale fin septembre ($)
data['tau']     = tau
data['z_alpha'] = z_alpha
data['M']       = 1_000_000     # big-M (doit être >> max quantité possible)
data['V_max']   = V_max_val     # capacité stockage globale (m³)

# ── Matières Premières ────────────────────────────────────────────────────────
data['prix_MP']       = {i: v["prix"]       for i, v in donnees_MP.items()}
data['stock_init_MP'] = {i: v["stock_init"] for i, v in donnees_MP.items()}
data['LT']            = {i: v["LT"]         for i, v in donnees_MP.items()}
data['MOQ']           = {i: v["MOQ"]        for i, v in donnees_MP.items()}
data['volume_MP']     = volume_MP
data['SS_MP']         = SS_MP   # stocks de sécurité calculés à l'étape 5.2

# ── Semi-Finis ────────────────────────────────────────────────────────────────
data['prix_SF']       = {j: v["prix"]       for j, v in donnees_SF.items()}
data['stock_init_SF'] = {j: v["stock_init"] for j, v in donnees_SF.items()}
data['ct_SF']         = {j: v["ct"]         for j, v in donnees_SF.items()}
data['MLS_SF']        = {j: v["MLS"]        for j, v in donnees_SF.items()}
data['charge_SF']     = {j: v["charge"]     for j, v in donnees_SF.items()}
data['volume_SF']     = volume_SF
data['SS_SF']         = SS_SF

# ── Produits Finis ────────────────────────────────────────────────────────────
data['prix_PF']       = {l: v["prix"]       for l, v in donnees_PF.items()}
data['stock_init_PF'] = {l: v["stock_init"] for l, v in donnees_PF.items()}
data['ct_PF']         = {l: v["ct"]         for l, v in donnees_PF.items()}
data['MLS_PF']        = {l: v["MLS"]        for l, v in donnees_PF.items()}
data['charge_PF']     = {l: v["charge"]     for l, v in donnees_PF.items()}
data['volume_PF']     = volume_PF
data['SS_PF']         = SS_PF

# ── Demande client, pipeline SAP, BOM, capacités ─────────────────────────────
data['demande'] = {(l, t): donnees_PF[l]["demande"][t]
                   for l in donnees_PF for t in [1, 2, 3, 4]}
data['OC']      = {(i, t): OC[i][t]     for i in OC     for t in [1, 2, 3, 4]}
data['WIP_SF']  = {(j, t): WIP_SF[j][t] for j in WIP_SF for t in [1, 2, 3, 4]}
data['WIP_PF']  = {(l, t): WIP_PF[l][t] for l in WIP_PF for t in [1, 2, 3, 4]}
data['a']       = {(i, j): a_bom[i][j]  for i in a_bom  for j in a_bom[i] if a_bom[i][j] > 0}
data['b']       = {(j, l): b_bom[j][l]  for j in b_bom  for l in b_bom[j] if b_bom[j][l] > 0}
data['Cap_SF']  = Cap_SF
data['Cap_PF']  = Cap_PF

# ── Création de l'instance concrète ──────────────────────────────────────────
instance = model.create_instance(data)

print("=" * 60)
print("  INSTANCE CONCRÈTE CRÉÉE")
print("=" * 60)
print(f"  Modèle       : {instance.name}")
print(f"  Périodes     : {list(noms_periodes.values())}")
print(f"  TARGET       : {value(instance.TARGET):>14,.0f} $")
print(f"  V_max        : {value(instance.V_max):>14,.0f} m³")
print(f"  M (big-M)    : {value(instance.M):>14,.0f}")
print(f"  tau          : {value(instance.tau)*100:.0f} %")
print(f"  z_alpha      : {value(instance.z_alpha)}")
print("=" * 60)


---
### 5.4 Résolution (appel réel de `resoudre`, défini à l'étape 4)

In [ ]:
resultats = resoudre(instance, solveur="cbc", verbose=False)

print("=" * 55)
print("  STATUT DE LA RÉSOLUTION")
print("=" * 55)
print(f"  Statut solveur      : {resultats.solver.termination_condition}")
print(f"  Valeur objectif (Z) : {value(instance.cost):>14,.2f} $")
print("=" * 55)


---
### 5.5 Vérification de la cible fiscale (C5)

In [ ]:
T_max = value(instance.T)

# Valeur totale du stock en fin d'horizon (fin septembre)
valeur_fin_sept = (
    sum(instance.prix_MP[i] * value(instance.S_MP[i, T_max]) for i in instance.MP)
  + sum(instance.prix_SF[j] * value(instance.S_SF[j, T_max]) for j in instance.SF)
  + sum(instance.prix_PF[l] * value(instance.S_PF[l, T_max]) for l in instance.PF)
)

print(f"  Valeur stock fin {noms_periodes[T_max]:<9}: {valeur_fin_sept:>14,.0f} $")
print(f"  Cible fiscale (TARGET)        : {value(instance.TARGET):>14,.0f} $")
print()
if valeur_fin_sept <= value(instance.TARGET):
    print("  ✅ Cible atteinte !")
else:
    ecart = valeur_fin_sept - value(instance.TARGET)
    print(f"  ❌ Cible dépassée — Écart : {ecart:,.0f} $")


---
### 5.6 Extraction des résultats en DataFrames

In [ ]:
def extraire(var, refs, label):
    """Extrait une variable Pyomo en DataFrame (références x périodes)."""
    lignes = [
        {"Référence": r, "Période": noms_periodes[t], "t": t, label: value(var[r, t])}
        for r in refs for t in instance.PER
    ]
    df = pd.DataFrame(lignes)
    display(df.pivot(index="Référence", columns="Période", values=label)[
        [noms_periodes[t] for t in instance.PER]
    ])
    return df

print("Stocks MP  :") ; df_S_MP = extraire(instance.S_MP, instance.MP, "Stock MP")
print("Stocks SF  :") ; df_S_SF = extraire(instance.S_SF, instance.SF, "Stock SF")
print("Stocks PF  :") ; df_S_PF = extraire(instance.S_PF, instance.PF, "Stock PF")
print("Commandes O_MP  :") ; df_O_MP = extraire(instance.O_MP, instance.MP, "O_MP")
print("Production P_SF :") ; df_P_SF = extraire(instance.P_SF, instance.SF, "P_SF")
print("Production P_PF :") ; df_P_PF = extraire(instance.P_PF, instance.PF, "P_PF")


---
### 5.7 Valeur de stock par période et graphiques

In [ ]:
# ── Tableau de valeur de stock par période et par étage ───────────────────────
valeurs = []
for t in instance.PER:
    v_mp = sum(instance.prix_MP[i] * value(instance.S_MP[i, t]) for i in instance.MP)
    v_sf = sum(instance.prix_SF[j] * value(instance.S_SF[j, t]) for j in instance.SF)
    v_pf = sum(instance.prix_PF[l] * value(instance.S_PF[l, t]) for l in instance.PF)
    valeurs.append({
        "Période": noms_periodes[t],
        "MP ($)": round(v_mp), "SF ($)": round(v_sf), "PF ($)": round(v_pf),
        "Total ($)": round(v_mp + v_sf + v_pf)
    })
df_valeur = pd.DataFrame(valeurs).set_index("Période")

print("Valeur de stock par période et par étage :")
display(df_valeur)


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))

# Courbe par étage
for col, color in [("MP ($)", "steelblue"), ("SF ($)", "darkorange"), ("PF ($)", "seagreen")]:
    ax.plot(df_valeur.index, df_valeur[col], marker="o", label=col, color=color)

# Courbe Total
ax.plot(df_valeur.index, df_valeur["Total ($)"],
        marker="s", linewidth=2.5, color="black", label="Total")

# Ligne cible fiscale
ax.axhline(value(instance.TARGET), color="red", linestyle="--", linewidth=1.5,
           label=f"Cible fiscale ({value(instance.TARGET):,.0f} $)")

# Annotation : valeur initiale
ax.scatter([df_valeur.index[0]], [val_total_init], color="black", zorder=5, s=80)
ax.annotate(f"Valeur initiale\n{val_total_init:,.0f} $",
            xy=(df_valeur.index[0], val_total_init),
            xytext=(0.1, 0.95), textcoords="axes fraction",
            fontsize=9, color="black")

ax.set_title("Trajectoire de la valeur de stock par étage — Juin à Septembre",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Période")
ax.set_ylabel("Valeur du stock ($)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


---
### 5.8 Résumé final

In [ ]:
print("=" * 60)
print("  RÉSUMÉ FINAL DE LA RÉSOLUTION")
print("=" * 60)
print(f"  Statut solveur          : {resultats.solver.termination_condition}")
print(f"  Valeur objectif Z       : {value(instance.cost):>14,.0f} $")
print(f"  Valeur stock fin {noms_periodes[T_max]:<9}: {valeur_fin_sept:>14,.0f} $")
print(f"  Cible fiscale (TARGET)  : {value(instance.TARGET):>14,.0f} $")
print(f"  Capacité stockage V_max : {value(instance.V_max):>14,.0f} m³")
print(f"  Écart cible             : {valeur_fin_sept - value(instance.TARGET):>+14,.0f} $")
print(f"  Cible respectée         : {'✅ Oui' if valeur_fin_sept <= value(instance.TARGET) else '❌ Non'}")
print("=" * 60)
